# Lab | Generative Adversarial Network (GAN) on CIFAR-10 Dataset
### Generate Fun Colorful Images: Animals, Cars, and More!

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model, datasets
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

In [ ]:
# Load and preprocess CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

# Normalize pixel values to [-1, 1] (better for GAN training)
x_train = (x_train.astype('float32') - 127.5) / 127.5
x_test = (x_test.astype('float32') - 127.5) / 127.5

print(f"Training data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")
print(f"Pixel value range: [{x_train.min():.2f}, {x_train.max():.2f}]")

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Set parameters
latent_dim = 100

In [ ]:
# Build the Generator Model
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(latent_dim,)),

        layers.Dense(4 * 4 * 256, use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Reshape((4, 4, 256)),  # 4x4x256

        layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),  # 8x8x128

        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),  # 16x16x64

        layers.Conv2DTranspose(3, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'),
        # 32x32x3, tanh matches the [-1, 1] normalized pixel range
    ], name="generator")
    return model


generator = build_generator()

In [ ]:
# Build the Discriminator Model
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),  # 16x16x64

        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),  # 8x8x128

        layers.Flatten(),
        # small L2 regularizer on the output layer - helps keep the discriminator
        # from overpowering the generator too quickly (see "Overpowering Discriminator" problem)
        layers.Dense(1, kernel_regularizer=regularizers.l2(1e-4)),
    ], name="discriminator")
    return model


discriminator = build_discriminator()

In [ ]:
# Define loss functions
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss


def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

In [ ]:
# Define optimizers
# beta_1=0.5 is the standard DCGAN recommendation - the default 0.9 tends to
# make GAN training oscillate/collapse
generator_optimizer = Adam(1e-4, beta_1=0.5)
discriminator_optimizer = Adam(1e-4, beta_1=0.5)

In [ ]:
# Training step function
@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], latent_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
# Training parameters
# NOTE: a GAN normally needs 50-100+ epochs on CIFAR-10 to produce anything
# recognizable. EPOCHS is kept small here on purpose, just to prove the training
# loop runs end-to-end - don't expect sharp images at 5 epochs.
EPOCHS = 5
BATCH_SIZE = 128
BUFFER_SIZE = x_train.shape[0]

train_dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

gen_losses = []
disc_losses = []

for epoch in range(EPOCHS):
    start = time.time()
    epoch_gen_loss = []
    epoch_disc_loss = []

    for image_batch in train_dataset:
        gen_loss, disc_loss = train_step(image_batch)
        epoch_gen_loss.append(float(gen_loss))
        epoch_disc_loss.append(float(disc_loss))

    gen_losses.append(np.mean(epoch_gen_loss))
    disc_losses.append(np.mean(epoch_disc_loss))

    print(f"Epoch {epoch + 1}/{EPOCHS} - gen_loss: {gen_losses[-1]:.4f} - "
          f"disc_loss: {disc_losses[-1]:.4f} - time: {time.time() - start:.1f}s")

## Exercise 1: Plot Training Losses

In [ ]:
### YOUR CODE HERE - Exercise 1 ###
plt.figure(figsize=(8, 5))
plt.plot(gen_losses, label='Generator loss')
plt.plot(disc_losses, label='Discriminator loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GAN Training Losses')
plt.legend()
plt.show()

## Exercise 2: Generate Image Grid

In [ ]:
### YOUR CODE HERE - Exercise 2 ###

grid_size = 8  # For 8x8 grid of images

noise = tf.random.normal([grid_size * grid_size, latent_dim])
generated_images = generator(noise, training=False)
generated_images = ((generated_images * 127.5) + 127.5).numpy().astype('uint8')

fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated_images[i])
    ax.axis('off')
plt.suptitle("Generated Images")
plt.tight_layout()
plt.show()

## Exercise 3: Compare Real vs Generated Images

In [ ]:
### YOUR CODE HERE - Exercise 3 ###

n_images = 4  # Number of images to compare

idx = np.random.choice(len(x_test), n_images, replace=False)
real_images = ((x_test[idx] * 127.5) + 127.5).astype('uint8')

noise = tf.random.normal([n_images, latent_dim])
fake_images = generator(noise, training=False)
fake_images = ((fake_images * 127.5) + 127.5).numpy().astype('uint8')

fig, axes = plt.subplots(2, n_images, figsize=(3 * n_images, 6))
for i in range(n_images):
    axes[0, i].imshow(real_images[i])
    axes[0, i].axis('off')

    axes[1, i].imshow(fake_images[i])
    axes[1, i].axis('off')

axes[0, 0].set_title("Real", loc='left')
axes[1, 0].set_title("Generated", loc='left')
plt.tight_layout()
plt.show()

## Exercise 4: Latent Space Interpolation

In [ ]:
### YOUR CODE HERE - Exercise 4 ###

n_steps = 10
z1 = tf.random.normal([1, latent_dim])
z2 = tf.random.normal([1, latent_dim])

alphas = np.linspace(0, 1, n_steps)
interpolated = tf.concat([(1 - a) * z1 + a * z2 for a in alphas], axis=0)

interp_images = generator(interpolated, training=False)
interp_images = ((interp_images * 127.5) + 127.5).numpy().astype('uint8')

fig, axes = plt.subplots(1, n_steps, figsize=(2 * n_steps, 2))
for i, ax in enumerate(axes):
    ax.imshow(interp_images[i])
    ax.axis('off')
plt.suptitle("Latent Space Interpolation (z1 -> z2)")
plt.tight_layout()
plt.show()

## Bonus Exercise: Model Summaries

In [ ]:
### YOUR CODE HERE - Bonus ###
# Display the architecture of your models
print("Generator Summary:")
generator.summary()

print("\nDiscriminator Summary:")
discriminator.summary()

gen_params = generator.count_params()
disc_params = discriminator.count_params()
print(f"\nGenerator parameters:     {gen_params:,}")
print(f"Discriminator parameters: {disc_params:,}")